## <b style="color: green;">Tools </b> ##

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")


In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")
model

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.7.7', 'langchain': '1.4.9', 'langchain-google-genai': '4.4.3'}}, profile={'name': 'Gemini 3.5 Flash Lite', 'release_date': '2026-07-21', 'last_updated': '2026-07-21', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True, 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'], 'reasoning_effort_default': 'minimal'}, google_api_key=SecretStr('**********'), model='gemini-3.5-flash-lite', temperature=None, client=<google.genai.client.Client object at 0x000002050533F380>, default_metadata=(), model_kwar

In [3]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"its sunny in {location}"

model_with_tools = model.bind_tools([get_weather])



In [6]:
response = model_with_tools.invoke("what is the weather like in boston")
print(response)

for tool_call in response.tool_calls:
    print(f"Tools:{tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content=[] additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}, '__gemini_function_call_thought_signatures__': {'call_10758470750469592624': 'EmAKXgFpFH0TxOutnHq5axRCxWpD00MkI0r2q+PBWrhR/Tc3D9rc97GU1p0tqfTVAVAFr5dvoC8+1XBxnSqt+hEmqSpxfk2RDOunSqdptReoEFH7z/Zr5jTmNdR1kDc46mM='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a1c506-9ac1-71f2-9158-eb2e4142268c-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'call_10758470750469592624', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 51, 'output_tokens': 16, 'total_tokens': 67, 'input_token_details': {'cache_read': 0}}
Tools:get_weather
Args: {'location': 'Boston'}


In [7]:
model_with_tools.invoke("hello")

AIMessage(content=[{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'EmAKXgFpFH0TQWKqbNBMY07AlpfqGoYGRq0nCJANi8lt4SmZlWf2B91SFnTOnDbAa13iWVbK8EQ57uZWR+v6mFyW5ucwJ9drDd6eYYeOnsJfbk+OpmCoiLwW2ey+WglpKBo='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1c507-8256-7316-ba61-b4a4c9143ce8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 45, 'output_tokens': 9, 'total_tokens': 54, 'input_token_details': {'cache_read': 0}})

In [10]:
# from langchain.agents import create_agent

# def get_weather(city:str)->str:
#     """Get the weather for a specified city"""
#     return f"The Weather in {city} is sunny."

# agent = create_agent(
#     model="gemini-3.5-flash-lite",
#     tools=[get_weather],
#     system_prompt="you are a helpful assistant"
# )
# agent.invoke("what is the weather in pune")

#### <b style="color: orange;">Tool Execution Loops </b> ####

In [11]:
model_with_tools.invoke("whats the weather in mumbai, do i need to carry umbrella?")

AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Mumbai"}'}, '__gemini_function_call_thought_signatures__': {'call_4541665011874026286': 'EmAKXgFpFH0TUtGd5Hj5tAc6OaxwjT2KTSnpizfSppOvRNOWaFc2DbwIVBuamHj1qB2S5tXESFs07uo4WYt6wFYRLXKqCQmKGEiHa+5S4fxfy2NBMY2d3pBSXuPrwMFICyw='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1c520-4dd8-72b5-b104-bb027f2c7598-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Mumbai'}, 'id': 'call_4541665011874026286', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 57, 'output_tokens': 16, 'total_tokens': 73, 'input_token_details': {'cache_read': 0}})

In [ ]:
messages = [{"role":"user","content":"Whats the weather in Mumbai,do i need to carry umbrealla ?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_result = get_weather(tool_call)
    # print(tool_result)
    messages.append(tool_result)
    # print(messages)

final_response = model_with_tools.invoke(messages)
print(final_response.text)

[{'role': 'user', 'content': 'Whats the weather in Mumbai,do i need to carry umbrealla ?'}, AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Mumbai"}'}, '__gemini_function_call_thought_signatures__': {'call_1213149911982025957': 'EmAKXgFpFH0THGOjcsMnrw/QPMRV4uBxE9H8uYSHAEz88OTvSCq1JoOKyUvRcCiOnVKwA2cpClg/jzCLW+85E2h906KvlrEWgRRv28upxHdonUGvXxPVTiC5/egRfTBX6DE='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1c52d-f842-70b7-8dbd-1d19934b5d50-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Mumbai'}, 'id': 'call_1213149911982025957', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 59, 'output_tokens': 16, 'total_tokens': 75, 'input_token_details': {'cache_read': 0}}), "The Weather in {'name': 'get_weather', 'args': {'location': 'Mumbai'}, 'id': 'call_1213149911982025957', 'typ

: 